# Convolutional Neural Network

### Importing the libraries

# Convolutional Neural Network

In [ ]:
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.preprocessing.image import ImageDataGenerator

tf.get_logger().setLevel('ERROR')

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('GPU tidak terdeteksi. Restart kernel, pilih Python (.venv cnn), lalu jalankan ulang dari cell pertama.')

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

mixed_precision.set_global_policy('mixed_float16')


In [ ]:
tf.__version__

In [ ]:
# Ubah parameter di sini untuk eksperimen GPU
# Variasi 1 laporan: filter awal 32 dan kernel 3x3.
IMAGE_SIZE = (180, 180)
BATCH_SIZE = 64
EPOCHS = 100
FILTERS = 32        # Variasi 1
KERNEL_SIZE = 3     # Kernel 3x3
L2_REG = 0.00001
CHECKPOINT_PATH = 'results/best_model_v1_f32_k3_180.keras'
PRED_IMAGE_PATHS = ['single_prediction/cat_or_dog_1.jpg', 'single_prediction/cat_or_dog_2.jpg']
RANDOM_SEED = 42
tf.keras.utils.set_random_seed(RANDOM_SEED)

## Part 1 - Data Preprocessing

### Preprocessing the Training set

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.1,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=(0.85, 1.15),
    horizontal_flip=True,
    rotation_range=20,
    fill_mode='nearest'
)
training_set = train_datagen.flow_from_directory('training_set',
                                                 target_size=IMAGE_SIZE,
                                                 batch_size=BATCH_SIZE,
                                                 class_mode='binary')

### Preprocessing the Test set

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_set = test_datagen.flow_from_directory('test_set',
                                            target_size=IMAGE_SIZE,
                                            batch_size=BATCH_SIZE,
                                            class_mode='binary',
                                            shuffle=False)

### Initialising the CNN

In [ ]:
cnn = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=[IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
])

### Step 1 - Convolution

In [ ]:
# Block 1: double conv (VGG-style) — 2 conv sebelum pooling, fitur lebih kaya
cnn.add(tf.keras.layers.Conv2D(
    filters=FILTERS, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)
))
cnn.add(tf.keras.layers.Conv2D(
    filters=FILTERS, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)
))
cnn.add(tf.keras.layers.BatchNormalization())

### Step 2 - Pooling

In [ ]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a second convolutional layer

In [ ]:
# Block 2: FILTERS*2, double conv
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 2, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 2, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Block 3: FILTERS*4, double conv
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Block 4: FILTERS*4, double conv
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Block 5: fitur lebih dalam, tapi feature map sudah kecil sehingga masih ramah GPU
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 8, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 8, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3 - Flattening

In [ ]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

In [ ]:
cnn.add(tf.keras.layers.Dense(units=256, activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Dropout(0.45))
cnn.add(tf.keras.layers.Dense(units=128, activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Dropout(0.25))

### Step 5 - Output Layer

In [ ]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid', dtype='float32'))

## Part 3 - Training the CNN

### Compiling the CNN

In [ ]:
# Learning rate dibuat lebih halus untuk training lebih panjang.
optimizer = tf.keras.optimizers.Adam(learning_rate=0.00025)
cnn.compile(optimizer = optimizer, loss = 'binary_crossentropy', metrics = ['accuracy'])

In [ ]:
import os
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

os.makedirs('results', exist_ok=True)

early_stop = EarlyStopping(monitor='val_accuracy', mode='max', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', mode='max', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
checkpoint = ModelCheckpoint(CHECKPOINT_PATH, monitor='val_accuracy', mode='max', save_best_only=True, verbose=1)

history = cnn.fit(x=training_set, validation_data=test_set, epochs=EPOCHS, callbacks=[early_stop, reduce_lr, checkpoint])

best_epoch = history.history['val_accuracy'].index(max(history.history['val_accuracy'])) + 1
train_acc_pct = history.history['accuracy'][best_epoch - 1] * 100
val_acc_pct = max(history.history['val_accuracy']) * 100
print(f'Best epoch: {best_epoch}')
print(f'Best train acc: {train_acc_pct:.2f}%')
print(f'Best val acc: {val_acc_pct:.2f}%')
print(f'Best model saved to: {CHECKPOINT_PATH}')

## Part 4 - Making a single prediction

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

class_indices = training_set.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}

print('class_indices:', class_indices)
for pred_path in PRED_IMAGE_PATHS:
    test_image = image.load_img(pred_path, target_size = IMAGE_SIZE)
    test_image = image.img_to_array(test_image)
    test_image = np.expand_dims(test_image, axis = 0)
    test_image = test_image / 255.0

    result = cnn.predict(test_image, verbose = 0)
    probability = float(result[0][0])
    predicted_index = 1 if probability >= 0.5 else 0
    predicted_label = idx_to_class[predicted_index]

    file_name = pred_path.split('/')[-1]
    print(f"{file_name}: prob={probability:.4f}, pred={predicted_label}")

In [ ]:
# Dinonaktifkan: output prediksi sekarang sudah dicetak per file di cell sebelumnya
